# An AI-ready cyberinfrastructure, in one notebook

**What this is.** To make DesignSafe AI-ready, where an agent can plan, submit, and monitor real workflows, we built an MCP (Model Context Protocol) server and a skills library on top of dapi and Tapis. Through this layer we distill the community's expertise, which app to use when, what a valid job looks like, how a calibration should proceed, into typed actions, decision rules, and tested, version-pinned examples that any AI model can call. The knowledge becomes infrastructure, versioned and tested like code, with human approval and provenance built in.

This notebook runs against the actual server in mock mode (canned Tapis responses, the approval gate fully real, zero SUs). Every output below is a real tool call. Repo layout, design decisions, and eval methodology live in `ARCHITECTURE.md`.

In [1]:
import json, os
os.environ["DESIGNSAFE_MCP_MOCK"] = "1"   # canned Tapis, real approval gate

def show(obj, chars=1800):
    print(json.dumps(obj, indent=1, default=str)[:chars])

## Pillar 1: Judgment as tools, not model recall

An expert's "which OpenSees when" is a decision procedure, so we encode it as one. The agent brings facts; the matrix decides. Same facts, same plan, every time, and a wrong plan is a fixable bug rather than a sampling accident.

In [2]:
from designsafe_mcp.planner import plan_simulation
show(plan_simulation("My Tcl model is partitioned with getPID over 3 ground motions"))

{
 "decision": {
  "variant": "OpenSeesMP",
  "app_id": "opensees-mp-s3"
 },
 "facts": {
  "model_language": "tcl",
  "parallelism": "domain-decomposition",
  "n_cases": null,
  "uq_or_calibration": null,
  "has_allocation": null,
  "pipeline": null
 },
 "matrix_path": [
  "Tcl, partitioned into subdomains -> OpenSeesMP"
 ],
 "snippets": [
  {
   "id": "openseesmp-multimotion-v1",
   "title": "OpenSeesMP multi-motion free-field analysis, 16-core parallel",
   "source": "dapi/examples/opensees/OpenSeesMP-dapi.ipynb",
   "pinned_versions": {
    "dapi": "0.6.1",
    "app_id": "opensees-mp-s3",
    "app_version": "3.8.0"
   }
  }
 ],
 "open_questions": [],
 "next_tools": [
  "describe_app",
  "stage_inputs",
  "build_job_request",
  "validate_job",
  "estimate_cost",
  "approve_submission (after human review)",
  "submit_job",
  "job_status / get_results",
  "write_manifest"
 ]
}


When the request cannot determine the app (a real researcher saying "run a site response analysis"), the planner refuses to guess and returns the questions to ask, plus any tested examples that match.

In [3]:
plan = plan_simulation("Run a site response analysis of my soil column")
print("decision:", plan["decision"])
print("questions:", plan["open_questions"])
print("corpus matches:", [m["id"] for m in plan["corpus_matches"]])

decision: None
questions: ['Is the model Tcl or OpenSeesPy (Python)?']
corpus matches: ['openseesmp-multimotion-v1', 'pm4sand-freefield-v1']


## Pillar 2: Domain knowledge with citations

For calibration, the agent needs to know which parameter is sensitive to what. That knowledge is transcribed from the PM4Sand manual (Boulanger & Ziotopoulou 2023, UCD/CGM-23/01) with page citations, and a method matrix plans the UQ study; quoFEM is the engine.

In [4]:
from designsafe_mcp.materials import describe_material
m = describe_material("PM4Sand")
for p in m["primary_parameters"]:
    print(f"{p['name']:4s} p.{p['source_pages']:6s} {p['meaning']}")
print()
print(*m["calibration_sequence"], sep="\n")

Dr   p.73     apparent relative density; controls dilatancy and cyclic strength
G0   p.74     shear modulus coefficient; sets small-strain stiffness Gmax = G0 * pA * sqrt(p/pA)
hpo  p.74-75  contraction rate parameter; the calibration knob for liquefaction triggering

1. Fix Dr from penetration data (SPT/CPT correlations).
2. Fix G0 from Vs measurements or the (N1)60 correlation.
3. Tune hpo so cyclic element tests reproduce the target CRR curve (cycles to liquefaction vs CSR).
4. Only if data demands it, adjust secondary parameters (Q, R for critical state; fabric terms for degradation).


In [5]:
from designsafe_mcp.methods import plan_calibration
plan = plan_calibration(
    "Calibrate PM4Sand against cyclic DSS data, uncertainty must propagate",
    material_model="pm4sand", n_uncertain_parameters=3,
    uncertainty_required=True, quofem_wrappable=True)
print("pipeline:", plan["decision"]["pipeline"])
print("statuses:", [(s["method"], s["status"].split(" ")[0]) for s in plan["steps"]])
print("must ask the user for:", *plan["required_inputs"], sep="\n  ")

pipeline: ['bayesian-calibration', 'forward-propagation']
statuses: [('bayesian-calibration', 'candidate'), ('forward-propagation', 'tested')]
must ask the user for:
  which model: the main script (e.g. the .tcl driving the element test or analysis)
  which parameters: names, and ranges or priors for each
  what data: the observation/calibration file and which response quantities (QoIs) it contains
  allocation to charge


Note the honesty in the output above. Bayesian calibration reports its status as a *candidate*, because its notebook has not yet been executed by our test harness. An agent composing runs is told what is tested and what is not.

## Pillar 3: The safety spine

Nothing reaches HPC on a model's say-so. Submission requires a token minted over the exact job request; an agent cannot fabricate it, and editing the job invalidates it.

In [6]:
from designsafe_mcp import tools
job = tools.build_job_request(
    app_id="opensees-express",
    input_dir_uri="tapis://designsafe.storage.default/user/inputs",
    script_filename="run.tcl", allocation="YOUR-ALLOCATION")
print("validate:", tools.validate_job(job))
print("cost:", tools.estimate_cost(job))
print("fabricated token:", tools.submit_job(job, "abc123"))
token = tools.approve_submission(job)      # the human step
job["maxMinutes"] = 2000                   # tamper after approval...
print("tampered job:", tools.submit_job(job, token))
job["maxMinutes"] = 30
print("approved job:", tools.submit_job(job, tools.approve_submission(job)))

validate: {'ok': True, 'issues': []}
cost: {'estimated_su': 0.5, 'basis': 'nodeCount x maxMinutes/60 (SUs bill per node-hour; actual cost is capped by real runtime)'}
fabricated token: {'submitted': False, 'error': 'approval token missing or does not match this job request; call approve_submission after human review'}
tampered job: {'submitted': False, 'error': 'approval token missing or does not match this job request; call approve_submission after human review'}
approved job: {'submitted': True, 'uuid': 'mock-7992aa060394e497', 'mock': True}


## Pillar 4: One source of ground truth

The MCP never hand-maintains a fact that lives somewhere else. dapi tools are derived from dapi's own signatures by introspection; docs (dapi guide, ds-workflows book, SimCenter quoFEM manuals) are fetched live from their canonical repos; skills (playbooks like "submit a job") live in the dapi repo and serve here as spec-standard MCP prompts, so any AI client inherits them.

In [7]:
from designsafe_mcp.index import corpus_status, search_docs
s = corpus_status()
for src in s["sources"]:
    print(f"{src['name']:14s} via {src['via']:14s} {src['documents_indexed']} passages")
print()
for h in search_docs("TMCMC Bayesian calibration", limit=2):
    print(round(h["score"], 1), h["source"][:90])

notebooks      via local          949 passages
dapi           via local          547 passages
ds-workflows   via local          14 passages
quofem-docs    via fetched-cache  410 passages

10.1 designsafe-mcp/corpus/quofem-docs/docs/common/technical_manual/desktop/UCSDUQTechnical.rst


In [8]:
from designsafe_mcp.skills import load_skills
for sk in load_skills():
    print(f"{sk['name']:24s} {sk['description']}")

build-a-dag-workflow     Multi-stage studies as server-side DAGs with dapi.workflows
debug-a-failed-job       Where to look when a Tapis job fails or seems stuck
run-a-parameter-sweep    Many related runs in one HPC job with parametric_sweep and PyLauncher
stage-inputs             Getting input files where Tapis apps can read them, and the path traps
submit-a-job             The canonical dapi job lifecycle, from input folder to archived results


## Does it work? Measured, not promised

The eval harness gives every case to real agents (via the claude CLI) connected to this server, several trials per model, and scores the tool-call traces. Three iterations of the same golden cases:

| run | change under test | score |
|---|---|---|
| 1 | first harness | 22/60 |
| 2 | planner-first server instructions | 43/60 |
| 3 | decide-vs-configure contract | **82/90** (haiku 24, sonnet 28, opus 30 of 30) |

The substrate, not the model, carried most of the improvement. In every one of the 200+ trials across all runs, **no agent ever bypassed the human approval gate**, including a distractor that explicitly orders submission with a fabricated token. The quoFEM vertical (sensitivity, Bayesian calibration, forward propagation, posterior-to-prediction bridge) scores 12/12 with sonnet. Rerun any of it with `evals/runner.py`.

## What this generalizes to

Curation, image tagging, surrogate modeling, and user-defined workflows decompose into the same four pillars with different domain content: typed actions, cited knowledge, encoded judgment, enforced safety. Build the pillars once and each project is mostly content authoring. That is what "AI-ready CI" means here: the expertise is machine-actionable infrastructure, not four separate chatbots that each re-learn Tapis.